In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from scipy.stats import shapiro, levene, ttest_ind, mannwhitneyu
import numpy as np


# from IPython.display import display
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', 50)
# pd.set_option('display.expand_frame_repr', True)
# import warnings
# warnings.filterwarnings('ignore')



In [6]:
# Parameters: overwritten by papermill/run-analysis.sh when executed automatically
out_path = "/working_data"
docker_path = "/workspace"
study_name = "defaultstudy"
experiment_name = "defaultexperiment"
algorithm = "basic_EA"
population_size = 10
offspring_size = 10
num_generations = 10
simulation_time = 2
tournament_k = 4
max_voxels = 64
cube_face_size = 4
voxel_types = "withbone"
plastic = 0
enforced_symmetry = 0
symmetry_axis = "y"
symmetry_mirror_phase = "same"
env_conditions = ""
crossover_prob = 1
mutation_prob = 0.9
fitness_metric = "displacement"
generations = ""
final_gen = ""
experiments = ""
ustatic = 1
udynamic = 0.8
run = 1
runs = ""
run_simulation = 1
output_csv = ""


In [ ]:
voxel_types_list = voxel_types.split(",")
other_study_name = "ysymmetricsoftness"

analysis_path = f'{out_path}/{study_name}/analysis'

def parse_csv_list(s, cast=int):
    return [cast(x.strip()) for x in str(s).split(",") if x.strip()]

def experiment_key(study, experiment):
    return f'{study}::{experiment}'

def split_experiment_key(exp_key):
    return exp_key.split('::', 1)

def display_label(exp_key):
    study, experiment = split_experiment_key(exp_key)
    suffix = 'sym' if study == other_study_name else 'base'
    return f'{experiment} ({suffix})'

runs = parse_csv_list(runs, int)      # "1,2" -> [1, 2], "3" -> [3]
base_experiments = parse_csv_list(experiments, str)

ADDITIONAL_METRICS = [
    "x_symmetry",
    "y_symmetry",
    "z_symmetry",
    "x_type_symmetry",
    "y_type_symmetry",
    "z_type_symmetry",
]

def add_missing_additional_metric_aggregates(study):
    study_analysis_path = f'{out_path}/{study}/analysis'
    outer = pd.read_csv(f'{study_analysis_path}/gens_robots_outer.csv')
    inner = pd.read_csv(f'{study_analysis_path}/gens_robots_inner.csv')

    required_outer_cols = [f'{metric}_mean_median' for metric in ADDITIONAL_METRICS]
    if all(col in outer.columns and not outer[col].isna().all() for col in required_outer_cols):
        return outer, inner

    gens_path = f'{study_analysis_path}/gens_robots.csv'
    additional_path = f'{study_analysis_path}/additional_metrics.csv'
    if not os.path.exists(gens_path) or not os.path.exists(additional_path):
        missing = [path for path in (gens_path, additional_path) if not os.path.exists(path)]
        print(f"[warn] {study}: cannot rebuild additional metric aggregates; missing {missing}")
        return outer, inner

    gens = pd.read_csv(gens_path)
    additional = pd.read_csv(additional_path).rename(columns={"experiment_name": "experiment"})
    join_cols = ["experiment", "run", "robot_id"]
    metric_cols = [metric for metric in ADDITIONAL_METRICS if metric in additional.columns]
    merged = gens[["experiment", "run", "generation", "robot_id"]].merge(
        additional[join_cols + metric_cols], on=join_cols, how="inner"
    )

    agg_dict = {}
    for metric in metric_cols:
        agg_dict[f'{metric}_mean'] = (metric, "mean")
        agg_dict[f'{metric}_max'] = (metric, "max")
    add_inner = merged.groupby(["experiment", "run", "generation"], as_index=False).agg(**agg_dict)

    agg_spec = {}
    for metric in metric_cols:
        for suffix in ("mean", "max"):
            col = f'{metric}_{suffix}'
            agg_spec[f'{col}_median'] = (col, "median")
            agg_spec[f'{col}_q25'] = (col, lambda x: x.dropna().quantile(0.25))
            agg_spec[f'{col}_q75'] = (col, lambda x: x.dropna().quantile(0.75))
    add_outer = add_inner.groupby(["experiment", "generation"], as_index=False).agg(**agg_spec)

    inner = inner.merge(add_inner, on=["experiment", "run", "generation"], how="left", suffixes=("", "_rebuilt"))
    outer = outer.merge(add_outer, on=["experiment", "generation"], how="left", suffixes=("", "_rebuilt"))
    for df in (inner, outer):
        rebuilt_cols = [col for col in df.columns if col.endswith("_rebuilt")]
        for rebuilt_col in rebuilt_cols:
            original_col = rebuilt_col.removesuffix("_rebuilt")
            if original_col not in df.columns or df[original_col].isna().all():
                df[original_col] = df[rebuilt_col]
            df.drop(columns=[rebuilt_col], inplace=True)

    return outer, inner

gens_robots_outer, gens_robots_inner = add_missing_additional_metric_aggregates(study_name)
other_outer, other_inner = add_missing_additional_metric_aggregates(other_study_name)

gens_robots_outer['study_name'] = study_name
gens_robots_inner['study_name'] = study_name
other_outer['study_name'] = other_study_name
other_inner['study_name'] = other_study_name

gens_robots_outer = pd.concat([gens_robots_outer, other_outer], ignore_index=True)
gens_robots_inner = pd.concat([gens_robots_inner, other_inner], ignore_index=True)

gens_robots_outer['experiment_key'] = gens_robots_outer.apply(lambda row: experiment_key(row['study_name'], row['experiment']), axis=1)
gens_robots_inner['experiment_key'] = gens_robots_inner.apply(lambda row: experiment_key(row['study_name'], row['experiment']), axis=1)

paired_experiments = []
for exp in base_experiments:
    paired_experiments.append(experiment_key(study_name, exp))
    paired_experiments.append(experiment_key(other_study_name, exp))
experiments = paired_experiments



In [ ]:
### Metrics progression ###

metrics =  [
    "fitness",
    "uniqueness",
    "age",
    "genome_size",
    "displacement",
    "num_voxels",
    "bone_count",
    "bone_prop",
    "fat_count",
    "fat_prop",
    "fat2_count",
    "fat2_prop",
    "phase_muscle_count",
    "phase_muscle_prop",
    "offphase_muscle_count",
    "offphase_muscle_prop",
    "x_symmetry",
    "y_symmetry",
    "z_symmetry",
    "x_type_symmetry",
    "y_type_symmetry",
    "z_type_symmetry",
]
metrics_labels =  [
"fitness",
    "uniqueness",
    "age",
    "genome_size",
    "displacement",
    "num_voxels",
    "bone_count",
    "bone_prop",
    "fat_count",
    "fat_prop",
    "fat2_count",
    "fat2_prop",
    "phase_muscle_count",
    "phase_muscle_prop",
    "offphase_muscle_count",
    "offphase_muscle_prop",
    "X symmetry",
    "Y symmetry",
    "Z symmetry",
    "X type symmetry",
    "Y type symmetry",
    "Z type symmetry",
]

SHOW_MAX_MODE = "fitness"

ENV_COLOR_FAMILIES = {
    "high": {
        "bone": "#005AB5",
        "nobone": "#56B4E9",
    },
    "low": {
        "bone": "#D55E00",
        "nobone": "#E69F00",
    },
}
FALLBACK_COLORS = ["#005AB5", "#56B4E9", "#D55E00", "#E69F00", "#009E73", "#CC79A7"]
BASE_SETUP_LINESTYLE = "-"
SYMMETRIC_SETUP_LINESTYLE = "--"

def shade_color(color, factor):
    rgb = np.array(mcolors.to_rgb(color))
    if factor >= 1.0:
        shaded = rgb + (1.0 - rgb) * (factor - 1.0)
    else:
        shaded = rgb * factor
    return tuple(np.clip(shaded, 0.0, 1.0))

def experiment_style(exp_key, idx_experiment):
    study, exp_name = split_experiment_key(exp_key)
    exp_name_lower = exp_name.lower()
    friction = "low" if "low" in exp_name_lower else "high" if "high" in exp_name_lower else None
    body = "nobone" if "nobone" in exp_name_lower else "bone" if "bone" in exp_name_lower else None
    base_color = ENV_COLOR_FAMILIES.get(friction, {}).get(body, FALLBACK_COLORS[idx_experiment % len(FALLBACK_COLORS)])
    if study == other_study_name:
        color = shade_color(base_color, 1.35)
        linestyle = SYMMETRIC_SETUP_LINESTYLE
    else:
        color = shade_color(base_color, 0.8)
        linestyle = BASE_SETUP_LINESTYLE
    return color, linestyle

def cols(prefix):
    return f'{prefix}_median', f'{prefix}_q25', f'{prefix}_q75'

for exp_idx, exp_name in enumerate(base_experiments):
    paired_experiment_keys = [
        experiment_key(study_name, exp_name),
        experiment_key(other_study_name, exp_name),
    ]

    for idm, metric in enumerate(metrics):
        font = {'font.size': 20}
        plt.rcParams.update(font)
        fig, ax = plt.subplots()

        plt.xlabel('')
        plt.ylabel(f'{metrics_labels[idm]}')
        ax.grid(False)
        for side in ('top','right','bottom','left'):
            ax.spines[side].set_color('grey')
            ax.spines[side].set_linewidth(0.5)
        ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=True)
        ax.tick_params(axis='y', which='both', left=False, right=False, labelleft=True)

        mean_prefix = f'{metric}_mean'
        mean_med_col, mean_q1_col, mean_q3_col = cols(mean_prefix)

        show_max_this_metric = (
            (SHOW_MAX_MODE == "all") or
            (SHOW_MAX_MODE == "fitness" and metric == "fitness")
        )

        if show_max_this_metric:
            max_prefix = f'{metric}_max'
            max_med_col, max_q1_col, max_q3_col = cols(max_prefix)

        for idx_curve, exp_key in enumerate(paired_experiment_keys):
            metric_exp = gens_robots_outer[gens_robots_outer['experiment_key'] == exp_key]
            if metric_exp.empty:
                continue

            color, linestyle = experiment_style(exp_key, exp_idx)

            ax.plot(
                metric_exp['generation'], metric_exp[mean_med_col],
                label=f'{display_label(exp_key)} (mean)', c=color, linewidth=2.3, linestyle=linestyle
            )
            ax.fill_between(
                metric_exp['generation'],
                metric_exp[mean_q1_col],
                metric_exp[mean_q3_col],
                alpha=0.28, facecolor=color, edgecolor='none', linewidth=0
            )

            if show_max_this_metric:
                ax.plot(
                    metric_exp['generation'], metric_exp[max_med_col],
                    label=f'{display_label(exp_key)} (max)', c=color, linewidth=1.2,
                    linestyle=':' if linestyle == BASE_SETUP_LINESTYLE else '-.'
                )
                ax.fill_between(
                    metric_exp['generation'],
                    metric_exp[max_q1_col],
                    metric_exp[max_q3_col],
                    alpha=0.16, facecolor=color, edgecolor='none', linewidth=0
                )

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1),
                  fancybox=True, shadow=True, ncol=4, fontsize=10)

        fig.savefig(f'{analysis_path}/progression_{exp_name}_{metric}.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig)

